# Day05 下午个人项目：电商用户多维分析

**姓名：** 请填写  
**专题方向：** A / B / C / D / E

本Notebook由每名学生独立完成，并随个人项目仓库提交到GitHub。

> 请只修改标有 `TODO` 的区域，不要删除任务说明、检查点、结论区和提交检查。

## 一、实验目标与提交要求

你需要独立完成：

1. 读取并验收第4天清洗后的数据；
2. 计算公共基础指标；
3. 选择一个专题完成单维分析；
4. 完成至少一个双维度交叉分析；
5. 输出三个标准CSV报表；
6. 撰写至少3条结论、1条限制和1项建议；
7. 将Notebook和输出文件提交到个人GitHub仓库。

### 必须遵守的分析边界

- 一行数据代表一名用户，不是一笔订单；
- `CustomerID`是标识符，不适合求平均值；
- `CashbackAmount`是返现金额，不是消费金额或销售额；
- 当前数据没有订单金额和订单日期，不能计算GMV、客单价或时间趋势；
- 分组差异只能说明关联，不能直接证明因果关系；
- 所有比例表必须同时包含样本量。

## 二、专题方向

| 专题 | 推荐字段 | 参考业务问题 |
|---|---|---|
| A 用户生命周期 | `TenureGroup` | 不同生命周期用户的流失和订单行为有何差异？ |
| B 投诉与服务体验 | `Complain`、`SatisfactionScore` | 投诉、满意度与流失存在怎样的关联？ |
| C 品类与订单行为 | `PreferedOrderCat` | 不同偏好品类用户的规模和订单行为有何差异？ |
| D 支付与优惠行为 | `PreferredPaymentMode` | 支付偏好与优惠行为是否存在分组差异？ |
| E 城市与设备行为 | `CityTier`、`PreferredLoginDevice` | 城市、设备与用户活跃或流失有何关联？ |

请选择一个专题作为单维分析主线。双维分析可以在此基础上增加另一个业务维度。

## 任务0：个人配置与运行环境

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)


# =========================
# TODO：填写个人信息与专题
# =========================
STUDENT_NAME = "文鹏瑞"
TOPIC = "A"


pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


def find_workspace_root(start=None):
    """从当前目录向上寻找项目根目录。"""
    start = Path.cwd() if start is None else Path(start)

    for candidate in [start, *start.parents]:
        data_path = (
            candidate
            / "output"
            / "day04_project"
            / "ecommerce_customer_cleaned.csv"
        )

        if data_path.exists():
            return candidate

    raise FileNotFoundError(
        "未找到清洗后数据，请检查："
        "output/day04_project/ecommerce_customer_cleaned.csv"
    )


ROOT = find_workspace_root()
DATA_PATH = (
    ROOT
    / "output"
    / "day04_project"
    / "ecommerce_customer_cleaned.csv"
)
OUTPUT_DIR = ROOT / "output" / "day05_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)
print("输入数据：", DATA_PATH)
print("输出目录：", OUTPUT_DIR)

姓名： 文鹏瑞
专题： A
输入数据： C:\Users\output\day04_project\ecommerce_customer_cleaned.csv
输出目录： C:\Users\output\day05_analysis


In [2]:
# 检查点0：个人信息与专题配置

assert STUDENT_NAME != "请填写姓名", "请填写STUDENT_NAME"
assert STUDENT_NAME.strip(), "姓名不能为空"

TOPIC = TOPIC.strip().upper()
assert TOPIC in {"A", "B", "C", "D", "E"}, \
    "TOPIC只能填写A、B、C、D或E"

expected_output_dir = ROOT / "output" / "day05_analysis"
assert OUTPUT_DIR == expected_output_dir, \
    "输出目录应为output/day05_analysis"

print("检查点0通过")
print("姓名：", STUDENT_NAME)
print("专题：", TOPIC)

检查点0通过
姓名： 文鹏瑞
专题： A


### 检查点0完成标志

- [ ] 已填写姓名；
- [ ] `TOPIC`只填写A、B、C、D或E；
- [ ] 输出目录为`output/day05_analysis`；
- [ ] Notebook文件名保持为`day05_pm_student_project.ipynb`。

## 任务1：读取并验收数据（必做）

In [3]:
# 读取第4天清洗后的数据
df = pd.read_csv(DATA_PATH)

print("数据形状：", df.shape)
display(df.head())
print("\n字段类型：")
display(df.dtypes.to_frame("数据类型"))

数据形状： (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60



字段类型：


,数据类型
CustomerID,int64
Churn,int64
Tenure,float64
PreferredLoginDevice,object
CityTier,int64
WarehouseToHome,float64
PreferredPaymentMode,object
Gender,object
HourSpendOnApp,float64
NumberOfDeviceRegistered,int64


In [4]:
# TODO 1：定义需要验收的核心字段
core_cols = [
    # 示例："CustomerID",
]


# TODO 2：完成数据验收表
# 至少包含：行数、列数、CustomerID重复数、核心字段缺失数、Churn取值
validation = None


# TODO 3：展示验收结果
# display(validation)

In [6]:
# 1. 读取数据
df = pd.read_csv(DATA_PATH)

# 2. 输出基本信息
print(f"数据形状: {df.shape}")
display(df.head())
display(df.dtypes)

# 1. 读取CSV（优先处理BOM头，再处理错误行）
df = pd.read_csv(DATA_PATH, encoding='utf-8-sig', on_bad_lines='warn')

# 2. 打印列数与列名，定位缺失列
print(f"读取后列数：{df.shape[1]}")
print(f"读取后列名：{df.columns.tolist()}")

# 3. 若列数仍为21，尝试强制指定列数（假设原始22列）
if df.shape[1] == 21:
    # 方案：读取为字典，再转换为DataFrame（避免自动截断）
    df_dict = pd.read_csv(DATA_PATH, encoding='utf-8-sig', header=0, dtype=str)
    df = pd.DataFrame(df_dict)
    print(f"强制读取后列数：{df.shape[1]}")
    print(f"强制读取后列名：{df.columns.tolist()}")

# 4. 生成缺失的TenureGroup列（若仍缺失）
core_cols = [
    "CustomerID", "Churn", "Tenure", "TenureGroup", "OrderCount",
    "CouponUsed", "CashbackAmount", "DaySinceLastOrder", "Complain",
    "PreferedOrderCat", "PreferredPaymentMode"
]
if 'Tenure' in df.columns and 'TenureGroup' not in df.columns:
    df['TenureGroup'] = pd.cut(
        df['Tenure'], 
        bins=[0, 12, 24, float('inf')], 
        labels=['短期用户', '中期用户', '长期用户']
    )
    print("已生成TenureGroup列，当前列数：", df.shape[1])

# 5. 验证列数是否匹配
print(f"最终列数：{df.shape[1]}（预期22列）")
# 3. 定义核心字段
core_cols = ["CustomerID", "Churn", "Tenure", "OrderCount", "CashbackAmount", 
             "HourSpendOnApp", "SatisfactionScore", "DaySinceLastOrder","TenureGroup"]

# 4. 构建验收表
validation = pd.DataFrame({
    "验收项": ["数据形状", "CustomerID重复数", "核心字段缺失数", "Churn取值"],
    "期望结果": ["(5630, 22)", "0", "0", "0和1"],
    "实际结果": [
        str(df.shape),
        df["CustomerID"].duplicated().sum(),
        df[core_cols].isna().sum().sum(),
        sorted(df["Churn"].unique().tolist())
    ]
})
display(validation)

# 5. 说明数据粒度
print("【数据粒度说明】当前数据一行代表一名用户。CustomerID是用户唯一标识，"
      "属于分类变量，对其求平均值在业务上毫无意义。")
df.columns = df.columns.str.strip()

数据形状: (5630, 20)


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.00,Mobile Phone,3,6.00,Debit Card,Female,3.00,3,Laptop & Accessory,2,Single,9,1,11.00,1.00,1.00,5.00,159.93
1,50002,1,9.00,Mobile Phone,1,8.00,UPI,Male,3.00,4,Mobile Phone,3,Single,7,1,15.00,0.00,1.00,0.00,120.90
2,50003,1,9.00,Mobile Phone,1,30.00,Debit Card,Male,2.00,4,Mobile Phone,3,Single,6,1,14.00,0.00,1.00,3.00,120.28
3,50004,1,0.00,Mobile Phone,3,15.00,Debit Card,Male,2.00,4,Laptop & Accessory,5,Single,8,0,23.00,0.00,1.00,3.00,134.07
4,50005,1,0.00,Mobile Phone,1,12.00,Credit Card,Male,3.00,3,Mobile Phone,5,Single,3,0,11.00,1.00,1.00,3.00,129.60


CustomerID                       int64
Churn                            int64
Tenure                         float64
PreferredLoginDevice            object
CityTier                         int64
WarehouseToHome                float64
PreferredPaymentMode            object
Gender                          object
HourSpendOnApp                 float64
NumberOfDeviceRegistered         int64
PreferedOrderCat                object
SatisfactionScore                int64
MaritalStatus                   object
NumberOfAddress                  int64
Complain                         int64
OrderAmountHikeFromlastYear    float64
CouponUsed                     float64
OrderCount                     float64
DaySinceLastOrder              float64
CashbackAmount                 float64
dtype: object

读取后列数：20
读取后列名：['CustomerID', 'Churn', 'Tenure', 'PreferredLoginDevice', 'CityTier', 'WarehouseToHome', 'PreferredPaymentMode', 'Gender', 'HourSpendOnApp', 'NumberOfDeviceRegistered', 'PreferedOrderCat', 'SatisfactionScore', 'MaritalStatus', 'NumberOfAddress', 'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed', 'OrderCount', 'DaySinceLastOrder', 'CashbackAmount']
已生成TenureGroup列，当前列数： 21
最终列数：21（预期22列）


,验收项,期望结果,实际结果
0,数据形状,"(5630, 22)","(5630, 21)"
1,CustomerID重复数,0,0
2,核心字段缺失数,0,508
3,Churn取值,0和1,"[0, 1]"


【数据粒度说明】当前数据一行代表一名用户。CustomerID是用户唯一标识，属于分类变量，对其求平均值在业务上毫无意义。


### 数据粒度说明

请用一句话说明一行数据代表什么：

> TODO：请填写。

请说明为什么`CustomerID`不能作为普通连续数值求平均：

> TODO：请填写。

## 任务2：公共基础指标（必做）

请构建`overall_metrics`，至少包含以下10项指标：

1. 用户数；
2. 流失人数；
3. 总体流失率；
4. 平均订单数；
5. 订单数中位数；
6. 平均优惠券使用次数；
7. 平均返现；
8. 平均App使用时长；
9. 平均满意度；
10. 平均距上次下单天数。

输出建议使用“指标、数值”两列的DataFrame。

In [7]:
# TODO：计算公共基础指标

overall_metrics = None


# TODO：展示结果
# display(overall_metrics)

In [9]:
overall_metrics = pd.DataFrame({
    "指标": [
        "用户数", "流失人数", "总体流失率", "平均订单数", "订单数中位数",
        "平均优惠券使用次数", "平均返现", "平均App使用时长", 
        "平均满意度", "平均距上次下单天数"
    ],
    "数值": [
        len(df),
        df["Churn"].sum(),
        df["Churn"].mean(),
        df["OrderCount"].mean(),
        df["OrderCount"].median(),
        df["CouponUsed"].mean(),
        df["CashbackAmount"].mean(),
        df["HourSpendOnApp"].mean(),
        df["SatisfactionScore"].mean(),
        df["DaySinceLastOrder"].mean()
    ]
})

# 提取总体流失率供检查点使用
overall_churn_rate = df["Churn"].mean()

display(overall_metrics)
print(f"【数据现象描述】当前样本共有 {len(df)} 名用户，总体流失率为 {overall_churn_rate:.2%}，"
      f"用户平均订单数为 {df['OrderCount'].mean():.2f} 次。")

,指标,数值
0,用户数,"5,630.00"
1,流失人数,948.00
2,总体流失率,0.17
3,平均订单数,2.96
4,订单数中位数,2.00
5,平均优惠券使用次数,1.72
6,平均返现,177.22
7,平均App使用时长,2.93
8,平均满意度,3.07
9,平均距上次下单天数,4.46


【数据现象描述】当前样本共有 5630 名用户，总体流失率为 16.84%，用户平均订单数为 2.96 次。


### 公共指标初步观察

请写出一条总体数据现象。此处只描述数据，不解释原因。

> TODO：请填写，例如“当前样本共有……名用户，总体流失率为……”。

## 任务3：单维专题分析（必做）

根据所选专题确定一个主分组字段，并使用`groupby + agg`完成命名聚合。

最低要求：

- 必须包含“用户数”；
- 至少再包含3项业务指标；
- 如果包含流失率或占比，必须保留0～1原始小数用于导出；
- 按业务意义排序；
- 分组字段在`reset_index()`后应保留为普通列。

In [14]:
# 专题 A 主分组字段
segment_field = "TenureGroup"

# 单维分组聚合
segment_analysis = df.groupby(segment_field).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean"),
    平均返现=("CashbackAmount", "mean")
).reset_index()

# 按生命周期业务顺序排序（短期 -> 中期 -> 长期）
# 如果你的标签是中文，确保排序逻辑正确
order_map = {'短期用户': 0, '中期用户': 1, '长期用户': 2}
segment_analysis['sort_key'] = segment_analysis[segment_field].map(order_map)
segment_analysis = segment_analysis.sort_values('sort_key').drop('sort_key', axis=1)

display(segment_analysis)

# 检查点：用户数总和必须等于 5630
print(f"分组用户数总和: {segment_analysis['用户数'].sum()}")

,TenureGroup,用户数,流失人数,流失率,平均订单数,平均返现
0,短期用户,3226,581,0.18,2.72,164.11
1,中期用户,1467,95,0.06,3.70,204.92
2,长期用户,429,0,0.00,3.55,222.34


分组用户数总和: 5122


,TenureGroup,投诉状态,用户数,流失人数,流失率,平均订单数,样本提示
0,短期用户,无投诉,2355,264,0.11,2.69,可观察
1,短期用户,有投诉,871,317,0.36,2.77,可观察
2,中期用户,无投诉,1053,43,0.04,3.85,可观察
3,中期用户,有投诉,414,52,0.13,3.35,可观察
4,长期用户,无投诉,304,0,0.00,3.75,可观察
5,长期用户,有投诉,125,0,0.00,3.06,可观察


双维组合用户数总和: 5122


### 单维专题分析记录

**本专题要回答的业务问题：**

> TODO：请填写。

**数据现象：**

> TODO：必须写明群体、用户数、指标和具体数值。

**可能解释：**

> TODO：使用“相关、可能、值得关注、需验证”等有边界语言。

## 任务4：双维度交叉分析（必做）

从以下维度中选择两个不同字段：

- `TenureGroup`
- `Complain`
- `PreferedOrderCat`
- `CityTier`
- `PreferredLoginDevice`
- `PreferredPaymentMode`

最低要求：

- 输出两个分组维度；
- 输出用户数、流失人数、流失率和至少1项行为指标；
- 将用户数少于30的组合标记为“小样本”，其余标记为“可观察”；
- 不得只展示流失率而省略用户数。

In [15]:
# 双维字段：生命周期 + 投诉状态
dim1 = "TenureGroup"
dim2 = "Complain"

# 双维分组聚合
cross_analysis = df.groupby([dim1, dim2]).agg(
    用户数=("CustomerID", "nunique"),
    流失人数=("Churn", "sum"),
    流失率=("Churn", "mean"),
    平均订单数=("OrderCount", "mean")
).reset_index()

# 添加样本提示列（<30 为小样本）
cross_analysis["样本提示"] = cross_analysis["用户数"].apply(
    lambda x: "小样本" if x < 30 else "可观察"
)

# 添加业务标签（0=无投诉，1=有投诉）
cross_analysis["投诉状态"] = cross_analysis[dim2].map({0: "无投诉", 1: "有投诉"})

# 调整列顺序，让报表更易读
cross_analysis = cross_analysis[[dim1, "投诉状态", "用户数", "流失人数", "流失率", "平均订单数", "样本提示"]]

display(cross_analysis)

# 检查点：组合用户数总和必须等于 5630
print(f"双维组合用户数总和: {cross_analysis['用户数'].sum()}")

,TenureGroup,投诉状态,用户数,流失人数,流失率,平均订单数,样本提示
0,短期用户,无投诉,2355,264,0.11,2.69,可观察
1,短期用户,有投诉,871,317,0.36,2.77,可观察
2,中期用户,无投诉,1053,43,0.04,3.85,可观察
3,中期用户,有投诉,414,52,0.13,3.35,可观察
4,长期用户,无投诉,304,0,0.00,3.75,可观察
5,长期用户,有投诉,125,0,0.00,3.06,可观察


双维组合用户数总和: 5122


### 双维分析记录

**最值得关注的维度组合：**

> TODO：请填写。

**该组合的用户数、流失率和比较对象：**

> TODO：请填写。

**是否存在小样本风险：**

> TODO：请填写，并说明判断依据。

**为什么不能直接写成因果结论：**

> TODO：请填写。

In [16]:

最值得关注的维度组合：
短期用户 且 有投诉（TenureGroup="短期用户" & Complain="有投诉"）

该组合的用户数、流失率和比较对象：
该组合共有 871 名用户，流失率高达 36%。其最直接的比较对象是“短期用户 且 无投诉”组合（2355人，流失率 11%）。有投诉的短期用户流失率是无投诉短期用户的 3.2 倍，差异极其显著。

是否存在小样本风险：
不存在小样本风险。判断依据：该组合的样本量达到了 871，在统计学上属于大样本，足以支撑结论的可靠性。相比之下，长期用户有投诉的样本量仅为 125，那个才需要警惕小样本风险。

为什么不能直接写成因果结论：
因为当前的交叉分析仅揭示了“投诉”与“高流失率”之间的强相关性，但无法证明因果关系。我们无法确定是“投诉行为本身”直接导致了用户流失，还是因为“产品体验差/服务糟糕”这一潜在的第三变量，同时引发了用户的投诉行为和最终的流失结果。要得出因果结论，需要进行 A/B 测试或更严谨的因果推断分析。



SyntaxError: invalid character '：' (U+FF1A) (684478280.py, line 1)

## 任务5：输出统计报表（必做）

In [17]:
# 输出三个标准CSV文件

outputs = {
    "overall_metrics.csv": overall_metrics,
    "segment_analysis.csv": segment_analysis,
    "cross_analysis.csv": cross_analysis,
}

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename
    table.to_csv(path, index=False, encoding="utf-8-sig")
    print("已输出：", path.relative_to(ROOT))

已输出： output\day05_analysis\overall_metrics.csv
已输出： output\day05_analysis\segment_analysis.csv
已输出： output\day05_analysis\cross_analysis.csv


In [18]:
# 检查点5：输出文件与回读验证

for filename, table in outputs.items():
    path = OUTPUT_DIR / filename

    assert path.exists(), f"缺少输出文件：{filename}"

    reloaded = pd.read_csv(path)

    assert reloaded.shape == table.shape, \
        f"{filename}回读后的形状与原表不一致"
    assert not any(
        str(col).startswith("Unnamed")
        for col in reloaded.columns
    ), f"{filename}包含多余索引列，请使用index=False导出"

    print(f"通过：{filename}，形状为{reloaded.shape}")

print("检查点5通过")

通过：overall_metrics.csv，形状为(10, 2)
通过：segment_analysis.csv，形状为(3, 6)
通过：cross_analysis.csv，形状为(6, 7)
检查点5通过


## 任务6：结论、限制与建议（必做）

### 结论1

在____用户中，____指标为____，与____相比____。对应证据表：____。

> TODO：请填写完整结论。

### 结论2

> TODO：请填写。

### 结论3

> TODO：请填写。

### 分析限制

至少写明一项当前数据不能支持的分析，或一项可能影响结论的限制。

> TODO：请填写。

### 运营建议与验证方式

提出一项与分析结果对应的建议，并说明还需要哪些数据或方法验证效果。

> TODO：请填写。

## 拓展任务（选做）

In [ ]:
# 可选方向：
# 1. 使用qcut或业务规则构建订单活跃度分层；
# 2. 将双维分析整理为第6天绘图使用的长表；
# 3. 对一个反直觉结果提出两种数据核查方法；
# 4. 增加一项不与必做任务重复的业务分析。

# TODO（选做）

## 最终检查：GitHub提交前验收

In [ ]:
required_files = [
    ROOT / "notebooks" / "day05_pm_student_project.ipynb",
    OUTPUT_DIR / "overall_metrics.csv",
    OUTPUT_DIR / "segment_analysis.csv",
    OUTPUT_DIR / "cross_analysis.csv",
]

missing_files = [
    str(path.relative_to(ROOT))
    for path in required_files
    if not path.exists()
]

assert not missing_files, \
    f"提交内容不完整，缺少文件：{missing_files}"

for csv_path in required_files[1:]:
    check_df = pd.read_csv(csv_path)
    assert not any(
        str(col).startswith("Unnamed")
        for col in check_df.columns
    ), f"{csv_path.name}仍包含多余索引列"

print("本地提交文件检查通过")
print("请重启内核并从头运行Notebook，然后提交并推送到个人GitHub仓库。")

### GitHub提交清单

- [ ] 已填写姓名和专题；
- [ ] Notebook已重启内核并从头运行成功；
- [ ] 所有检查点均已通过；
- [ ] `output/day05_analysis/`中包含三个CSV；
- [ ] CSV中没有`Unnamed`索引列；
- [ ] 至少完成3条结论、1条限制和1项建议；
- [ ] 没有把返现写成消费额；
- [ ] 没有把相关关系写成确定因果关系；
- [ ] 已提交并推送到个人GitHub仓库。

### 最终反思

1. 本次分析中最重要的数据发现是什么？
2. 哪个检查点最能帮助你发现错误？
3. 哪条结论最容易被误解为因果关系？
4. 如果增加一个字段，你最希望增加什么？
5. 第6天准备把哪张统计表转化为图表？为什么？